# DBSCAN Quick Start Guide

**Estimated Time:** 10 minutes  
**Difficulty:** Beginner

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand what DBSCAN is and when to use it
- Apply DBSCAN to a simple clustering problem
- Interpret DBSCAN results (clusters and noise points)
- Compare DBSCAN with K-Means clustering
- Understand the basic parameters (eps and min_pts)

## Prerequisites

- Basic Python programming
- Familiarity with NumPy and Matplotlib
- Basic understanding of clustering concepts
- No prior knowledge of DBSCAN required!

## Paper References

This notebook introduces concepts from:
- **Section 1** (p. 226): Introduction and motivation for density-based clustering
- **Section 4.1** (p. 227): Core concepts - epsilon-neighborhood, core points, noise
- **Section 4.2** (p. 228): DBSCAN algorithm overview

**Full Reference:**  
Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *KDD* (Vol. 96, No. 34, pp. 226-231).

**Paper Location:** `1996-DBSCAN-KDD.pdf` in repository root

---

## Part 1: What is DBSCAN?

**DBSCAN** stands for **D**ensity-**B**ased **S**patial **C**lustering of **A**pplications with **N**oise.

### Key Idea

DBSCAN groups together points that are closely packed (high density) and marks points in low-density regions as outliers (noise). Unlike K-Means, DBSCAN:

- **Doesn't require you to specify the number of clusters** in advance
- **Can find clusters of arbitrary shapes** (not just circular)
- **Automatically detects outliers** as noise points

### Two Simple Parameters

DBSCAN needs only two parameters:

1. **eps (ε)**: The maximum distance between two points to be considered neighbors
2. **min_pts**: The minimum number of points needed to form a dense region (cluster)

### Three Types of Points

DBSCAN classifies each point as one of three types:

- **Core Point**: Has at least `min_pts` neighbors within distance `eps`
- **Border Point**: Within `eps` distance of a core point, but doesn't have enough neighbors to be core
- **Noise Point**: Neither core nor border - an outlier!

---

## Part 2: Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import KMeans
import sys
sys.path.append('..')

from src.dbscan_from_scratch import DBSCAN
from src.visualization import DBSCANVisualizer

# Set random seed for reproducibility
np.random.seed(42)

# Initialize visualizer
visualizer = DBSCANVisualizer()

print("✓ All imports successful!")

---

## Part 3: Your First DBSCAN Clustering

Let's start with a simple example using the "moons" dataset - two crescent-shaped clusters.

In [ ]:
# Generate sample data: two crescent moons
X, _ = make_moons(n_samples=200, noise=0.05, random_state=42)

# Visualize the raw data
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, s=50)
plt.title('Original Data: Two Moons', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Dataset shape: {X.shape}")
print(f"Number of points: {len(X)}")

### Apply DBSCAN

Now let's apply DBSCAN with reasonable parameters:
- `eps=0.3`: Points within 0.3 units are considered neighbors
- `min_pts=5`: Need at least 5 points to form a dense region

In [ ]:
# Create and fit DBSCAN model
dbscan = DBSCAN(eps=0.3, min_pts=5)
labels = dbscan.fit_predict(X)

# Analyze results
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"\n📊 DBSCAN Results:")
print(f"  • Number of clusters found: {n_clusters}")
print(f"  • Number of noise points: {n_noise}")
print(f"  • Number of core points: {len(dbscan.get_core_points())}")

# Visualize clustering results
visualizer.plot_clusters(X, labels, title="DBSCAN Clustering Results (eps=0.3, min_pts=5)")
plt.show()

### Understanding the Results

In the plot above:
- **Different colors** represent different clusters
- **Black 'x' markers** are noise points (outliers)
- DBSCAN successfully identified the two crescent-shaped clusters!

Notice how DBSCAN handles the non-circular shape perfectly - this is one of its key strengths.

---

## Part 4: Visualizing Point Types

Let's visualize the three types of points DBSCAN identifies:

In [ ]:
# Visualize point type classification
visualizer.plot_point_types(
    X, 
    labels, 
    dbscan.core_sample_indices_, 
    eps=0.3,
    title="DBSCAN Point Type Classification"
)
plt.show()

print("\n📌 Point Type Breakdown:")
print(f"  • Core points (blue): Form the 'skeleton' of clusters")
print(f"  • Border points (green): On the edges of clusters")
print(f"  • Noise points (black): Outliers that don't belong to any cluster")

---

## Part 5: Comparing DBSCAN with K-Means

Let's see how DBSCAN compares to K-Means on the same dataset.

### Why This Comparison Matters

K-Means is one of the most popular clustering algorithms, but it has limitations:
- Assumes clusters are spherical (circular)
- Requires you to specify the number of clusters (k)
- Assigns every point to a cluster (no outlier detection)

DBSCAN addresses all these limitations!

In [ ]:
# Apply K-Means with k=2 (we know there are 2 moons)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)

# Create side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DBSCAN results
for label in set(labels):
    if label == -1:
        mask = labels == label
        axes[0].scatter(X[mask, 0], X[mask, 1], c='black', marker='x', s=50, alpha=0.5, label='Noise')
    else:
        mask = labels == label
        axes[0].scatter(X[mask, 0], X[mask, 1], s=50, alpha=0.7, label=f'Cluster {label}')

axes[0].set_title('DBSCAN (eps=0.3, min_pts=5)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# K-Means results
for label in set(kmeans_labels):
    mask = kmeans_labels == label
    axes[1].scatter(X[mask, 0], X[mask, 1], s=50, alpha=0.7, label=f'Cluster {label}')

# Plot K-Means centroids
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                c='red', marker='*', s=300, edgecolors='black', linewidths=2, label='Centroids')

axes[1].set_title('K-Means (k=2)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Comparison Insights:")
print("  • DBSCAN: Correctly identifies the two crescent shapes")
print("  • K-Means: Struggles with non-circular shapes, splits clusters incorrectly")
print("  • DBSCAN: Detects outliers as noise")
print("  • K-Means: Forces every point into a cluster")

### When K-Means Works Better

Let's try a dataset where K-Means performs well: well-separated spherical clusters.

In [ ]:
# Generate well-separated spherical clusters
X_blobs, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.5, random_state=42)

# Apply both algorithms
dbscan_blobs = DBSCAN(eps=0.8, min_pts=5)
dbscan_labels_blobs = dbscan_blobs.fit_predict(X_blobs)

kmeans_blobs = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels_blobs = kmeans_blobs.fit_predict(X_blobs)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DBSCAN
for label in set(dbscan_labels_blobs):
    if label == -1:
        mask = dbscan_labels_blobs == label
        axes[0].scatter(X_blobs[mask, 0], X_blobs[mask, 1], c='black', marker='x', s=50, alpha=0.5)
    else:
        mask = dbscan_labels_blobs == label
        axes[0].scatter(X_blobs[mask, 0], X_blobs[mask, 1], s=50, alpha=0.7)

axes[0].set_title('DBSCAN on Spherical Clusters', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# K-Means
for label in set(kmeans_labels_blobs):
    mask = kmeans_labels_blobs == label
    axes[1].scatter(X_blobs[mask, 0], X_blobs[mask, 1], s=50, alpha=0.7)

axes[1].scatter(kmeans_blobs.cluster_centers_[:, 0], kmeans_blobs.cluster_centers_[:, 1], 
                c='red', marker='*', s=300, edgecolors='black', linewidths=2)

axes[1].set_title('K-Means on Spherical Clusters', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Both algorithms work well on spherical, well-separated clusters!")
print("   But DBSCAN still has the advantage of not requiring k in advance.")

### Algorithm Comparison Summary

| Feature | DBSCAN | K-Means |
|---------|--------|----------|
| **Cluster Shape** | Any shape | Spherical only |
| **Number of Clusters** | Automatic | Must specify k |
| **Outlier Detection** | Yes (noise points) | No |
| **Parameters** | eps, min_pts | k |
| **Deterministic** | Yes | No (random init) |
| **Best For** | Spatial data, arbitrary shapes, noisy data | Well-separated spherical clusters |
| **Complexity** | O(n²) or O(n log n) | O(nki) |

**When to use DBSCAN:**
- You don't know how many clusters to expect
- Your clusters have irregular shapes
- Your data contains outliers
- You're working with spatial/geographic data

**When to use K-Means:**
- You know the number of clusters
- Clusters are roughly spherical and well-separated
- You need fast clustering on large datasets
- No outliers in your data

---

## Part 6: Understanding Parameters

Let's experiment with different parameter values to understand their effect.

In [ ]:
# Generate test data
X_test, _ = make_moons(n_samples=200, noise=0.05, random_state=42)

# Try different eps values
eps_values = [0.1, 0.3, 0.5]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, eps in enumerate(eps_values):
    dbscan_test = DBSCAN(eps=eps, min_pts=5)
    labels_test = dbscan_test.fit_predict(X_test)
    
    n_clusters = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise = list(labels_test).count(-1)
    
    # Plot
    for label in set(labels_test):
        if label == -1:
            mask = labels_test == label
            axes[idx].scatter(X_test[mask, 0], X_test[mask, 1], c='black', marker='x', s=50, alpha=0.5)
        else:
            mask = labels_test == label
            axes[idx].scatter(X_test[mask, 0], X_test[mask, 1], s=50, alpha=0.7)
    
    axes[idx].set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} noise', fontsize=11)
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Parameter Effects:")
print("  • eps too small (0.1): Many small clusters, lots of noise")
print("  • eps just right (0.3): Correct clustering")
print("  • eps too large (0.5): Clusters merge together")

---

## Summary

Congratulations! You've completed the DBSCAN quick start guide. Here's what you learned:

### Key Takeaways

1. **DBSCAN is density-based**: It groups points in high-density regions and identifies outliers

2. **Two parameters**: 
   - `eps` (ε): Maximum distance for neighborhood
   - `min_pts`: Minimum points to form a dense region

3. **Three point types**:
   - Core points: Dense region centers
   - Border points: Cluster edges
   - Noise points: Outliers

4. **Advantages over K-Means**:
   - No need to specify number of clusters
   - Handles arbitrary cluster shapes
   - Detects outliers automatically
   - Deterministic results

5. **Best use cases**:
   - Spatial/geographic data
   - Unknown number of clusters
   - Non-spherical cluster shapes
   - Noisy datasets with outliers

### What You Can Do Now

✓ Apply DBSCAN to simple clustering problems  
✓ Interpret clustering results and identify noise  
✓ Compare DBSCAN with K-Means  
✓ Understand the effect of parameter choices  

---

## Next Steps

Ready to dive deeper? Here's your learning path:

### Immediate Next Steps

1. **[01_dbscan_basics.ipynb](01_dbscan_basics.ipynb)** - More detailed examples and datasets
2. **[02_eps_minpts_tuning.ipynb](02_eps_minpts_tuning.ipynb)** - Learn systematic parameter selection
3. **[docs/01_theory_and_math.md](../docs/01_theory_and_math.md)** - Mathematical foundations

### For Deeper Understanding

- **Theory**: Read `docs/02_density_concepts.md` for formal definitions
- **Algorithm**: Study `docs/03_algorithm_details.md` for step-by-step walkthrough
- **Implementation**: Explore `src/dbscan_from_scratch.py` to see the code
- **Applications**: Try `notebooks/04_spatial_data_clustering.ipynb` for real-world examples

### Advanced Topics

- Performance optimization with spatial indexing
- Handling high-dimensional data
- OPTICS algorithm (DBSCAN extension)
- Custom distance metrics

---

## Additional Resources

- **Original Paper**: `1996-DBSCAN-KDD.pdf` in repository root
- **Paper Reading Guide**: `docs/00_how_to_read_the_paper.md`
- **Complete Documentation**: `docs/` directory
- **Learning Roadmap**: `docs/learning_roadmap.md`

---

**Happy Clustering! 🎯**